In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import spherical_jn, factorial
from scipy.integrate import cumulative_trapezoid

# =====================================================
# Zinc Hartree-Fock STO data
# Zn: 1s(2) 2s(2) 2p(6) 3s(2) 3p(6) 3d(10) 4s(2) 
# Ground term: 4S
# Format: (n_j, Z_j, C_j)
# =====================================================

zinc_1s = [
    (1, 30.7026,  0.950023),
    (2, 26.0605,  0.055373),
    (2, 13.2539, -0.000597),
    (3, 37.7395,  0.007693),
    (3,  9.8590, -0.000202),
    (3,  6.6334, -0.000014),
    (3,  4.4668, -0.000007),
    (4, 15.6259,  0.000616),
    (4,  2.6384, -0.000001),
    (4,  1.6263,  0.000001),
    (4,  1.0391,  0.000000),
]

zinc_2s = [
    (1, 30.7026,  0.293360),
    (2, 26.0605,  0.203764),
    (2, 13.2539, -1.012933),
    (3, 37.7395,  0.000174),
    (3,  9.8590, -0.099869),
    (3,  6.6334,  0.005685),
    (3,  4.4668, -0.002788),
    (4, 15.6259, -0.093558),
    (4,  2.6384,  0.000306),
    (4,  1.6263, -0.000170),
    (4,  1.0391,  0.000058),
]

zinc_3s = [
    (1, 30.7026, -0.110788),
    (2, 26.0605, -0.090902),
    (2, 13.2539,  0.459095),
    (3, 37.7395, -0.000721),
    (3,  9.8590,  0.072788),
    (3,  6.6334, -0.664296),
    (3,  4.4668, -0.532662),
    (4, 15.6259,  0.116494),
    (4,  2.6384, -0.006730),
    (4,  1.6263,  0.001318),
    (4,  1.0391, -0.000334),
]

zinc_4s = [
    (1, 30.7026,  0.022191),
    (2, 26.0605,  0.018507),
    (2, 13.2539, -0.093638),
    (3, 37.7395,  0.000170),
    (3,  9.8590, -0.021745),
    (3,  6.6334,  0.171174),
    (3,  4.4668,  0.105412),
    (4, 15.6259, -0.025059),
    (4,  2.6384, -0.273189),
    (4,  1.6263, -0.542094),
    (4,  1.0391, -0.320316),
]

zinc_2p = [
    (2, 51.3224, -0.000384),
    (2, 19.2139, -0.314928),
    (2, 10.4311, -0.445010),
    (2,  5.0040, -0.007426),
    (3, 15.7917, -0.282257),
    (3,  4.0672,  0.000573),
    (3,  2.6701, -0.000295),
]

zinc_3p = [
    (2, 51.3224, -0.000097),
    (2, 19.2139, -0.066963),
    (2, 10.4311, -0.532060),
    (2,  5.0040,  0.823000),
    (3, 15.7917, -0.024274),
    (3,  4.0672,  0.413856),
    (3,  2.6701,  0.048684),
]

zinc_3d = [
    (3, 15.0303,  0.017396),
    (3,  8.1007,  0.230530),
    (3,  4.9123,  0.421574),
    (3,  2.8828,  0.372225),
    (3,  1.7108,  0.130294),
]
# =====================================================
# STO normalization constant
# N_j = sqrt((2Z)^(2n+1)/(2n)!)
# =====================================================
def N_sto(n, Z):
    return np.sqrt((2*Z)**(2*n + 1) / factorial(2*n, exact=False))

def R_nl(r, coeffs):
    R = np.zeros_like(r)
    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        R += C * N * r**(n-1) * np.exp(-Z*r)
    return R
# =====================================================
# Grids
# =====================================================

r = np.linspace(1e-6, 60.0, 20000)     # radial grid
p = np.arange(0.0, 1000.0 + 0.05, 0.05) # momentum grid
Q = np.arange(0.0, 100.0 + 0.05, 0.05) # Q grid

# =====================================================
# Calculate chi_nl(p)
# chi_nl(p) = sqrt(2/pi) int R_nl(r) j_l(pr) r^2 dr
# =====================================================

def chi_p(p_grid, r_grid, R_grid, l):
    chi = []

    for pp in p_grid:
        jl = spherical_jn(l, pp*r_grid)
        integrand = R_grid * jl * r_grid**2
        val = np.sqrt(2/np.pi) * np.trapezoid(integrand, r_grid)
        chi.append(val)

    return np.array(chi)

# =====================================================
# Calculate J(Q)
# I(p) = |chi(p)|^2 p^2
# J(Q) = 1/2 int_Q^inf I(p)/p dp
# =====================================================

def calculate_J(p_grid, chi_grid, Q_grid):
    I = chi_grid**2 * p_grid**2

    integrand = np.zeros_like(p_grid)
    integrand[1:] = I[1:] / p_grid[1:]

    # Reverse cumulative integral from p to infinity
    rev_integral = cumulative_trapezoid(
        integrand[::-1],
        p_grid[::-1],
        initial=0
    )

    J_p = -0.5 * rev_integral[::-1]

    # Interpolate J from p-grid to Q-grid
    J_Q = np.interp(Q_grid, p_grid, J_p)

    return I, J_Q

# =====================================================
# Build Zinc radial orbitals
# =====================================================
def print_R_formula(coeffs, name):
    print(f"\n{name}(r) =")

    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        A = C * N

        if n == 1:
            print(f"{A:+.4f} * exp(-{Z:.4f} r)")
        elif n == 2:
            print(f"{A:+.4f} * r * exp(-{Z:.4f} r)")
        elif n == 3:
            print(f"{A:+.4f} * r^2 * exp(-{Z:.4f} r)")
        elif n == 4:
            print(f"{A:+.4f} * r^3 * exp(-{Z:.4f} r)")
            
R1s = R_nl(r, zinc_1s)
R2s = R_nl(r, zinc_2s)
R3s = R_nl(r, zinc_3s)
R4s = R_nl(r, zinc_4s)
R2p = R_nl(r, zinc_2p)
R3p = R_nl(r, zinc_3p)
R3d = R_nl(r, zinc_3d)

print_R_formula(zinc_1s, "zn_R1s")
print_R_formula(zinc_2s, "zn_R2s")
print_R_formula(zinc_3s, "zn_R3s")
print_R_formula(zinc_4s, "zn_R4s")
print_R_formula(zinc_2p, "zn_R2p")
print_R_formula(zinc_3p, "zn_R3p")
print_R_formula(zinc_3d, "zn_R3d")

# =====================================================
# Momentum-space wavefunctions
# =====================================================

chi_1s = chi_p(p, r, R1s, l=0)
chi_2s = chi_p(p, r, R2s, l=0)
chi_3s = chi_p(p, r, R3s, l=0)
chi_4s = chi_p(p, r, R4s, l=0)
chi_2p = chi_p(p, r, R2p, l=1)
chi_3p = chi_p(p, r, R3p, l=1)
chi_3d = chi_p(p, r, R3d, l=2)

# =====================================================
# Momentum densities and Compton profiles
# =====================================================
I_1s, J_1s = calculate_J(p, chi_1s, Q)
I_2s, J_2s = calculate_J(p, chi_2s, Q)
I_3s, J_3s = calculate_J(p, chi_3s, Q)
I_4s, J_4s = calculate_J(p, chi_4s, Q)
I_2p, J_2p = calculate_J(p, chi_2p, Q)
I_3p, J_3p = calculate_J(p, chi_3p, Q)
I_3d, J_3d = calculate_J(p, chi_3d, Q)

print("Norm 1s =", np.trapezoid(I_1s, p))
print("Norm 2s =", np.trapezoid(I_2s, p))
print("Norm 2p =", np.trapezoid(I_2p, p))
print("Norm 3s =", np.trapezoid(I_3s, p))
print("Norm 3p =", np.trapezoid(I_3p, p))
print("Norm 3d =", np.trapezoid(I_3d, p))
print("Norm 4s =", np.trapezoid(I_4s, p))


# Zinc has  1s(2) 2s(2) 2p(6) 3s(2) 3p(6) 3d(10) 4s(2) 
J_total = 2*J_1s + 2*J_2s + 6*J_2p + 2*J_3s + 6*J_3p + 10*J_3d + 2*J_4s 

# =====================================================
# Print and save table
# =====================================================
table = pd.DataFrame({
    "Q": Q,
    "J_1s(2)": J_1s,
    "J_2s(2)": J_2s,
    "J_2p(6)": J_2p,
    "J_3s(2)": J_3s,
    "J_3p(6)": J_3p,
    "J_3d(10)":J_3d,
    "J_4s(2)": J_4s,
    "J_total_Zn": J_total
    })
print(table)
table.to_csv("Zn__STO_HF_profile.csv", index = False)

# =====================================================
# Plot radial orbitals
# =====================================================

plt.figure(figsize=(8,6))
plt.plot(r, R1s, label="R_1s")
plt.plot(r, R2s, label="R_2s")
plt.plot(r, R2p, label="R_2p")
plt.plot(r, R3s, label="R_3s")
plt.plot(r, R3p, label="R_3p")
plt.plot(r, R4s, label="R_4s")
plt.plot(r, R3d, label="R_3d")
plt.xlim(0, 2)
plt.xlabel("r (a.u.)")
plt.ylabel("R_nl(r)")
plt.title("Zn Hartree-Fock Radial Orbitals")
plt.grid(True)
plt.legend()
plt.show()

# =====================================================
# Plot Compton profiles
# =====================================================
plt.figure(figsize=(8,6))
plt.plot(Q, 2*J_1s, label="1s(2)")
plt.plot(Q, 2*J_2s, label="2s(2)")
plt.plot(Q, 6*J_2p, label="2p(6)")
plt.plot(Q, 2*J_3s, label="3s(2)")
plt.plot(Q, 6*J_3p, label="3p(6)")
plt.plot(Q, 2*J_4s, label="4s(2)")
plt.plot(Q, 10*J_3d, label="3d(10)")
plt.plot(Q, J_total, label="Zn total", linewidth=2)

plt.xlabel("Q (a.u.)")
plt.ylabel("J(Q)")
plt.title("Zn Compton Profile from HF STO Orbitals")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()



zn_R1s(r) =
+323.2413 * exp(-30.7026 r)
+221.6789 * r * exp(-26.0605 r)
-0.4409 * r * exp(-13.2539 r)
+1071.0777 * r^2 * exp(-37.7395 r)
-0.2563 * r^2 * exp(-9.8590 r)
-0.0044 * r^2 * exp(-6.6334 r)
-0.0006 * r^2 * exp(-4.4668 r)
+16.3591 * r^3 * exp(-15.6259 r)
-0.0000 * r^3 * exp(-2.6384 r)
+0.0000 * r^3 * exp(-1.6263 r)
+0.0000 * r^3 * exp(-1.0391 r)

zn_R2s(r) =
+99.8145 * exp(-30.7026 r)
+815.7436 * r * exp(-26.0605 r)
-748.0133 * r * exp(-13.2539 r)
+24.2256 * r^2 * exp(-37.7395 r)
-126.7023 * r^2 * exp(-9.8590 r)
+1.8020 * r^2 * exp(-6.6334 r)
-0.2214 * r^2 * exp(-4.4668 r)
-2484.6106 * r^3 * exp(-15.6259 r)
+0.0027 * r^3 * exp(-2.6384 r)
-0.0002 * r^3 * exp(-1.6263 r)
+0.0000 * r^3 * exp(-1.0391 r)

zn_R3s(r) =
-37.6951 * exp(-30.7026 r)
-363.9148 * r * exp(-26.0605 r)
+339.0246 * r * exp(-13.2539 r)
-100.3831 * r^2 * exp(-37.7395 r)
+92.3450 * r^2 * exp(-9.8590 r)
-210.5606 * r^2 * exp(-6.6334 r)
-42.3037 * r^2 * exp(-4.4668 r)
+3093.7197 * r^3 * exp(-15.6259 r)
-0.0597 * r^3

PermissionError: [Errno 13] Permission denied: 'Zn__STO_HF_profile.csv'